In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used
# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [(1.1732+1.3325)/2, 1.1732+1.3325] # First value is the average of the two photopeaks. They have the same branching ratio

# colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kAzure, ROOT.kSpring, ROOT.kPink, ROOT.kMagenta, ROOT.kTeal, ROOT.kYellow, ROOT.kViolet, ROOT.kCyan]
# colors = [ROOT.kAzure+9, ROOT.kOrange+1, ROOT.kRed+1, ROOT.kGray+1, ROOT.kViolet-5, ROOT.kRed-5, ROOT.kOrange+8, ROOT.kYellow-5, ROOT.kGray+2, ROOT.kCyan-8]
colors = [ROOT.kAzure-7, ROOT.kOrange+1, ROOT.kRed-7, ROOT.kCyan-5, ROOT.kGreen-5, ROOT.kOrange-4, ROOT.kMagenta-5, ROOT.kRed-9, ROOT.kOrange-7, ROOT.kGray+1]

In [2]:
files_A = sorted((project_root/'data/260519').glob('SUBT_A0[1-4]*.csv'))
files_B = sorted((project_root/'data/260519').glob('SUBT_B0[1-4]*.csv'))
print(files_A)
print(files_B)


[PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_A01_NewSourceTest_Background.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_A02_NewSourceTest_Cs137.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_A03_NewSourceTest_Na22.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_A04_NewSourceTest_Co60.csv')]
[PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B01_NewSourceTest_Background.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B02_NewSourceTest_Cs137.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B03_NewSourceTest_Na22.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B04_NewSourceTest_Co60.csv')]


In [3]:
acqs_A = [SiphraAcquisition(file) for file in files_A]
acqs_B = [SiphraAcquisition(file) for file in files_B]
print(acqs_A)
print(acqs_B)

[SIPHRAAcquisition(File: 'SUBT_A01_NewSourceTest_Background'), SIPHRAAcquisition(File: 'SUBT_A02_NewSourceTest_Cs137'), SIPHRAAcquisition(File: 'SUBT_A03_NewSourceTest_Na22'), SIPHRAAcquisition(File: 'SUBT_A04_NewSourceTest_Co60')]
[SIPHRAAcquisition(File: 'SUBT_B01_NewSourceTest_Background'), SIPHRAAcquisition(File: 'SUBT_B02_NewSourceTest_Cs137'), SIPHRAAcquisition(File: 'SUBT_B03_NewSourceTest_Na22'), SIPHRAAcquisition(File: 'SUBT_B04_NewSourceTest_Co60')]


In [4]:
# histograms
from collections import namedtuple

DataHandler = namedtuple('DataHandler', ['acq', 'hist', 'src', 'subt'])

hists = {}
sources = []
handlers = {}
for acq, crystal in zip([*acqs_A, *acqs_B], [*('A'*4), *('B'*4)]):
    filepath = acq.filepath.stem
    src = (MetadataLoader.load(acq.metadataFile)).source
    src_key = 'BG' if src == 'Background' else src[:2]
    sources.append(src)
    siphra_code = acq.filepath.stem[5]
    print(src)
    # Generate histograms
    hist = ROOT.TH1F(f"{src}_SIPHRA_{siphra_code}", "", BITS12, 0, BITS12)
    hist.Fill(acq['s']/len(acq.active_chs))
    hists[f'{crystal}_{src_key}'] = hist
    handlers[f'{crystal}_{src_key}'] = DataHandler(acq, hist, src, None)
    del hist

Background
Cs-137
Na-22
Co-60
Background
Cs-137
Na-22
Co-60


In [64]:
# Subtract backgrounds here if correction factors are not configured, checking each one

correction_factors = {
    'A_Cs': 0.7,
    'A_Na': 1.05,
    'A_Co': 1.2,
    'B_Cs': 1,
    'B_Na': 1.1,
    'B_Co': 1.25,
}

if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 800, 600)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgd = ROOT.TLegend(0.48, 0.61, 0.75, 0.83)

whichSiphra = 'B'
source = 'Cs'
bg = handlers[f'{whichSiphra}_BG']
data_key = f'{whichSiphra}_{source}'
sgnl = handlers[data_key]

bg_scaled = bg.hist.Clone()
correction_factor = correction_factors[data_key] if correction_factors[data_key] else 1
bg_scaled.Scale(correction_factor*sgnl.acq.exposure/bg.acq.exposure)

subtracted = sgnl.hist.Clone(f'{sgnl.src}_{whichSiphra}_Subt.')
subtracted.Add(bg_scaled, -1)
# subts[data_key].hist = subtracted

lgd.AddEntry(sgnl.hist, 'Signal')
lgd.AddEntry(bg_scaled, 'Background')
lgd.AddEntry(subtracted, 'Subtracted')
sgnl.hist.SetLineColor(colors[4])
bg_scaled.SetLineColor(colors[3])
subtracted.SetLineColor(colors[2])
sgnl.hist.SetTitle(f'{sgnl.src} SIPHRA {data_key[:1]}')
sgnl.hist.Draw('hist')
bg_scaled.Draw('hist sames')
subtracted.Draw('hist sames')
cv.SetLogy()
lgd.Draw()
cv.Draw()

new_handler = DataHandler(sgnl.acq, sgnl.hist, sgnl.src, subtracted)
handlers[data_key] = new_handler

In [11]:
handlers

{'A_BG': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A01_NewSourceTest_Background'), hist=<cppyy.gbl.TH1F object at 0x5f10dfc0f150>, src='Background', subt=None),
 'A_Cs': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A02_NewSourceTest_Cs137'), hist=<cppyy.gbl.TH1F object at 0x5f10e0143cd0>, src='Cs-137', subt=<cppyy.gbl.TH1F object at 0x5f10e28e5a10>),
 'A_Na': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A03_NewSourceTest_Na22'), hist=<cppyy.gbl.TH1F object at 0x5f10d96d21e0>, src='Na-22', subt=<cppyy.gbl.TH1F object at 0x5f10e211dc70>),
 'A_Co': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A04_NewSourceTest_Co60'), hist=<cppyy.gbl.TH1F object at 0x5f10da330620>, src='Co-60', subt=<cppyy.gbl.TH1F object at 0x5f10e1ff42e0>),
 'B_BG': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_B01_NewSourceTest_Background'), hist=<cppyy.gbl.TH1F object at 0x5f10df428990>, src='Background', subt=None),
 'B_Cs': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_B02_NewSourceTest_Cs137'), hist=<cppyy.g

In [12]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 1600, 600)
cv.Divide(2,1)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(2)]

srcs_list = list(handlers.values())
srcs_list.pop(4)
srcs_list.pop(0)


for n, handler in enumerate(srcs_list):
    cv.cd(n//3 + 1)
    hist = handler.subt
    lgds[n//3].AddEntry(hist, handler.src)
    if not n%3:
        s = 'A' if n < 3 else 'B'
        hist.SetTitle(f'SIPHRA {s}')
        hist.GetXaxis().SetTitle("ADC channel number")
        hist.GetYaxis().SetTitle("Relative counts")
        hist.SetLineColor(colors[(n%3)])
        hist.Draw('hist')
    else:
        hist.SetLineColor(colors[(n%3)])
        hist.Draw('hist sames')

for c in range(2):
    cv.cd(c+1)
    lgds[c].Draw()
    cv.cd(c+1).SetLogy()

cv.Draw()


In [13]:
a = 3
colors[-a]

623

In [14]:
NA22_MEV

[0.511, 1.2745, 1.7854999999999999]

In [15]:
# Fit ranges for each peak. Same order as in srcs_list
fit_ranges = {
    'A_Cs': [(107,205)], # Cs siphra A
    'A_Na': [(89,152),(254,349),(380,431)], # Na siphra A
    'A_Co': [(228,351),(503,587)], # Co siphra A
    'B_Cs': [(107,211)],
    'B_Na': [(85,167),(261,374),(406,477)],
    'B_Co': [(240,376),(525, 612)]
}

for k, ranges in fit_ranges.items():
    try:
        hist = handlers[k].subt
        source = handlers[k].src
        chip = k[0]
        # print(f'{k}: chip: {chip}, src: {source}')
    except:
        print(f'Unmatched key {k}')

    for n, r in enumerate(ranges):
        fit_fn = ROOT.TF1(f"{source}_gaus_{n+1}", "gaus", 3)
        fit_status = hist.Fit(fit_fn, 'L S +', "", r[0], r[1])

****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      113.898
Chi2                      =      227.796
NDf                       =           95
Edm                       =  4.16932e-07
NCalls                    =           84
Constant                  =      3254.08   +/-   9.47293     
Mean                      =      148.846   +/-   0.0803696   
Sigma                     =      28.2149   +/-   0.0844885    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      53.7001
Chi2                      =        107.4
NDf                       =           60
Edm                       =  4.65619e-09
NCalls                    =           93
Constant                  =      715.773   +/-   5.28682     
Mean                      =      121.212   +/-   0.237091    
Sigma                     =       27.487   +/-   0.39358      	 (limited)
****************************************
Minimizer i

In [16]:
handlers['A_Co'].subt.GetListOfFunctions().ls()

OBJ: TList	TList	Doubly linked list : 0
 OBJ: TPaveStats	stats  	X1= 3686.400140 Y1=3.632541 X2=4505.600134 Y2=4.022536
 OBJ: TF1	Co-60_gaus_1	gaus : 0 at: 0x5f10e2dba340
 OBJ: TF1	Co-60_gaus_2	gaus : 0 at: 0x5f10e2ce2880


In [17]:
# graph of position of photopeaks

def get_mean_and_error(func_name, hist):
    f = hist.GetFunction(func_name)
    idx = f.GetParNumber('Mean')
    return f.GetParameter(idx), f.GetParError(idx)

COLOR_A = colors[0]
COLOR_B = colors[1]

means_A, errors_A = zip(*[
    get_mean_and_error('Cs-137_gaus_1', handlers['A_Cs'].subt),
    get_mean_and_error('Na-22_gaus_1',  handlers['A_Na'].subt),
    get_mean_and_error('Na-22_gaus_2',  handlers['A_Na'].subt),
    get_mean_and_error('Na-22_gaus_3',  handlers['A_Na'].subt),
    get_mean_and_error('Co-60_gaus_1',  handlers['A_Co'].subt),
    get_mean_and_error('Co-60_gaus_2',  handlers['A_Co'].subt),
])

means_A  = np.array(means_A,  dtype=np.float64)
errors_A = np.array(errors_A, dtype=np.float64)

means_B, errors_B = zip(*[
    get_mean_and_error('Cs-137_gaus_1', handlers['B_Cs'].subt),
    get_mean_and_error('Na-22_gaus_1',  handlers['B_Na'].subt),
    get_mean_and_error('Na-22_gaus_2',  handlers['B_Na'].subt),
    get_mean_and_error('Na-22_gaus_3',  handlers['B_Na'].subt),
    get_mean_and_error('Co-60_gaus_1',  handlers['B_Co'].subt),
    get_mean_and_error('Co-60_gaus_2',  handlers['B_Co'].subt),
])

means_B  = np.array(means_B,  dtype=np.float64)
errors_B = np.array(errors_A, dtype=np.float64)

peaks_real = np.array([*CS137_MEV, *NA22_MEV, *CO60_MEV])

if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 800, 600)
lg = ROOT.TLegend(0.48, 0.61, 0.75, 0.83)

zeros = np.zeros(len(means_A), dtype=np.float64)

peaks_graph_A = ROOT.TGraph(6, means_A, peaks_real)
peaks_graph_A.SetMarkerColor(COLOR_A)
peaks_graph_A.SetMarkerStyle(20)
correction_B = 1
peaks_graph_B = ROOT.TGraph(6, means_B*correction_B, peaks_real)
peaks_graph_B.SetMarkerColor(COLOR_B)
peaks_graph_B.SetMarkerStyle(20)

peaks_graph_A.SetTitle("Position of the photopeaks of the radiation sources")
peaks_graph_A.GetXaxis().SetTitle("ADC channel number")
peaks_graph_A.GetYaxis().SetTitle("Energy (MeV)")

lg.AddEntry(peaks_graph_A, "SIPHRA A")
lg.AddEntry(peaks_graph_B, "SIPHRA B")
peaks_graph_A.Draw("AP")
peaks_graph_B.Draw("P same")
lg.Draw()
cv.Draw()

In [18]:
# Fitting the calibration curves

calib_A = ROOT.TF1('Calibration_A', 'pol1', means_A.min()-50, means_A.max()+50)
peaks_graph_A.Fit(calib_A, 'RS 0')

calib_B = ROOT.TF1('Calibration_B', 'pol1', means_B.min()-50, means_B.max()+50)
peaks_graph_B.Fit(calib_B, 'RS 0')



****************************************
Minimizer is Linear / Migrad
Chi2                      =    0.0106604
NDf                       =            4
p0                        =   -0.0835364   +/-   0.0491596   
p1                        =   0.00471022   +/-   0.000147809 
****************************************
Minimizer is Linear / Migrad
Chi2                      =    0.0134464
NDf                       =            4
p0                        =   -0.0563666   +/-   0.0543699   
p1                        =   0.00433306   +/-   0.000152789 


In [19]:
#### if ROOT.gROOT.FindObject('cv'):
ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 800, 600)
lg = ROOT.TLegend(0.15, 0.75, 0.36, 0.88)

lg.AddEntry(peaks_graph_A, "Detector A")
lg.AddEntry(peaks_graph_B, "Detector B")

calib_A.SetLineColor(COLOR_A)
calib_A.SetLineStyle(2)
calib_B.SetLineColor(COLOR_B)
calib_B.SetLineStyle(2)

peaks_graph_A.Draw("AP")
peaks_graph_B.Draw("P same")
calib_A.Draw("same")
calib_B.Draw("same")
lg.Draw()

# Fit parameters

# eqbox_A = ROOT.TPaveText(0.15, 0.65, 0.45, 0.85, "NDC")
# eqbox_A.SetFIllColor(0)
# eqbox_A.(1)

# get parameters
calibPars = {}
eqboxes = {}
for chip, fitfn in zip(['A', 'B'], [calib_A, calib_B]):
    eqboxes[f'{chip}'] = ROOT.TPaveText(0.15, 0.7, 0.48, 0.8, "NDC")
    box =  eqboxes[f'{chip}']
    box.SetFillColor(0)
    box.SetBorderSize(1)
    a = fitfn.GetParameter(0)
    b = fitfn.GetParameter(1)
    sa = fitfn.GetParError(0)
    sb = fitfn.GetParError(1)

    calibPars[f'a_{chip}'] = a
    calibPars[f'b_{chip}'] = b
    calibPars[f'sa_{chip}'] = sa
    calibPars[f'sb_{chip}'] = sb

    box.AddText(f"Detector {chip}")
    box.AddText(f"E = ({a:.4f} #pm {sa:.4f}) + ({b:.4f} #pm {sb:.4f}) x")
    box.Draw()


cv.Update()
cv.Draw()

# Dynamic range and verification

In [20]:
# Calibrate BG spectra

def calibrate(data, a, b):
    ''' Performs linear energy calibration (a + bx) over a data array containing ADC values.'''
    return a + b*data

data_bgA_calib = calibrate(handlers['A_BG'].acq['s']/16, calibPars['a_A'], calibPars['b_A'])
data_bgB_calib = calibrate(handlers['B_BG'].acq['s']/16, calibPars['a_B'], calibPars['b_B'])

hist_bgA_calib = ROOT.TH1F(f"BG_SIPHRA_A", "", BITS12, data_bgA_calib.min(), data_bgA_calib.max())
hist_bgA_calib.Fill(data_bgA_calib)
hist_bgB_calib = ROOT.TH1F(f"BG_SIPHRA_B", "", BITS12, data_bgB_calib.min(), data_bgB_calib.max())
hist_bgB_calib.Fill(data_bgB_calib)

In [21]:
print('Dynamic ranges:')
print(f'SIPHRA A: {(calibPars['a_A'] + calibPars['b_A']*BITS12):.4f} MeV')
print(f'SIPHRA B: {(calibPars['a_B'] + calibPars['b_B']*BITS12):.4f} MeV')

Dynamic ranges:
SIPHRA A: 19.2095 MeV
SIPHRA B: 17.6919 MeV


In [22]:
def hist_quickShow(hists, color_offset=0, tone_offset=0):
    # colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kAzure, ROOT.kSpring, ROOT.kPink, ROOT.kMagenta, ROOT.kTeal, ROOT.kYellow, ROOT.kViolet, ROOT.kCyan]
    colors = [ROOT.kAzure-7, ROOT.kOrange+1, ROOT.kRed-7, ROOT.kCyan-5, ROOT.kGreen-5, ROOT.kOrange-4, ROOT.kMagenta-5, ROOT.kRed-9, ROOT.kOrange-7, ROOT.kGray+1]
    l_colors = len(colors)
    if ROOT.gROOT.FindObject('cv'):
        ROOT.gROOT.FindObject('cv').Close()

    cv = ROOT.TCanvas('cv', 'cv', 800, 600)
    
    ROOT.gStyle.SetOptStat(11)
    ROOT.gStyle.SetStatFontSize(0.03)
    ROOT.gStyle.SetStatW(0.16)

    list(hists)
    counter = 0

    for hist in hists:
        hist.SetLineColor(colors[(counter%l_colors + color_offset)%l_colors] + counter//l_colors + tone_offset)
        if counter == 0:
            hist.Draw('hist')
        else:
            hist.Draw('hist sames')
        counter += 1
    cv.SetLogy()
    cv.Draw()
    return cv

hist_quickShow([hist_bgA_calib, hist_bgB_calib], color_offset=2, tone_offset=0)

In [23]:
import importlib
if 'analysis' in sys.modules:
    del sys.modules['analysis']

from analysis import fit_peak_expbg

print(f'{K40_MEV[0]=}    {TL208_MEV[0]=}')

fit_peak_expbg(hist_bgA_calib, 'K_peak', 1.1, 2, 315, 1.5, 0.3, showFit=True, keep_prev_fncs=True)
fit_peak_expbg(hist_bgA_calib, 'Tl_peak', 2.25, 2.75, 41, 2.6, 0.3, showFit=True, keep_prev_fncs=True)

fit_peak_expbg(hist_bgB_calib, 'K_peak', 1.15, 2, 315, 1.5, 0.3, showFit=True, keep_prev_fncs=True)
fit_peak_expbg(hist_bgB_calib, 'Tl_peak', 2.25, 2.75, 41, 2.6, 0.3, showFit=True, keep_prev_fncs=True)

K40_MEV[0]=1.46    TL208_MEV[0]=2.614


(<cppyy.gbl.TF1 object at 0x5f10e2f88450>,
 <cppyy.gbl.TFitResultPtr object at 0x5f10e35acd70>)

****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      113.401
Chi2                      =      226.802
NDf                       =          193
Edm                       =  5.73256e-08
NCalls                    =          449
Const                     =      8.01652   +/-   0.0468389   
Decay                     =     -2.16582   +/-   0.0352595   
Norm                      =       196.39   +/-   3.09887     
Mean                      =      1.50409   +/-   0.00219136  
Sigma                     =     0.124595   +/-   0.00253122  
****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      40.9885
Chi2                      =      81.9769
NDf                       =          105
Edm                       =  2.30607e-06
NCalls                    =         1019
Const                     =     0.166879   +/-   0.642479    
Decay                     =      1.34163   +/-   0.267652    
Norm          

In [24]:
bgHists = [hist_bgA_calib, hist_bgB_calib]
bgPeaks_real = [K40_MEV, TL208_MEV]
kpeaks = [bgHists[0].GetFunction('K_peak').GetParameter('Mean'), bgHists[1].GetFunction('K_peak').GetParameter('Mean')]
tlpeaks = [bgHists[0].GetFunction('Tl_peak').GetParameter('Mean'), bgHists[1].GetFunction('Tl_peak').GetParameter('Mean')]
errs = []
errs = [[(expr - real[0])/real[0] for expr in l_exp] for real, l_exp in zip(bgPeaks_real, [kpeaks, tlpeaks])]
print("Errors in the energy reconstruction of the background K-40 and Tl-208 photopeaks")
print(f'{'-'*50}')
print(f'{'Source':^10} | {'SIPHRA A':^15} | {'SIPHRA B':^15}')
print(f'{'-'*50}')
print(f'{'K-40':^10} | {f'{errs[0][0]*100:.3f} %':>15} | {f'{errs[0][1]*100:.3f} %':>15}')
print(f'{'Tl-208':^10} | {f'{errs[1][0]*100:.3f} %':>15} | {f'{errs[1][1]*100:.3f} %':>15}')
print(f'{'-'*50}')

Errors in the energy reconstruction of the background K-40 and Tl-208 photopeaks
--------------------------------------------------
  Source   |    SIPHRA A     |    SIPHRA B    
--------------------------------------------------
   K-40    |         3.020 % |         4.517 %
  Tl-208   |         7.089 % |         5.605 %
--------------------------------------------------


# Energy resolution

For the energy resolution, we can get the calibrated data from the fitted data of the spectra we previously obtained. For a linear calibration $E = a + b \cdot x$, the transform for the parameters of the fitted gaussians is:

**Mean**

$\mu_E = a + b\cdot \mu_x$

**Sigma**

$\sigma_E = b \cdot \sigma_x$

The propagated error in the sigma is

$\sigma^2_{\sigma_E} = b^2\sigma^2_{\sigma_x} + \sigma^2_x\sigma^2_b + 2b\sigma_x \mathrm{cov}(\sigma_x, b)$

The last term is negligible since $\sigma_x$ comes from the histogram fit and $b$ from the calibration, so they are independent.

In [58]:
def get_sigma_and_error(func_name, hist):
    f = hist.GetFunction(func_name)
    idx = f.GetParNumber('Sigma')
    return f.GetParameter(idx), f.GetParError(idx)

COLOR_A = colors[0]
COLOR_B = colors[1]



sigmas_A, errs_sig_A = zip(*[
    get_sigma_and_error('Cs-137_gaus_1', handlers['A_Cs'].subt),
    get_sigma_and_error('Na-22_gaus_1',  handlers['A_Na'].subt),
    get_sigma_and_error('Na-22_gaus_2',  handlers['A_Na'].subt),
    get_sigma_and_error('Na-22_gaus_3',  handlers['A_Na'].subt),
    get_sigma_and_error('Co-60_gaus_1',  handlers['A_Co'].subt),
    get_sigma_and_error('Co-60_gaus_2',  handlers['A_Co'].subt),
])

sigmas_B, errs_sig_B = zip(*[
    get_sigma_and_error('Cs-137_gaus_1', handlers['B_Cs'].subt),
    get_sigma_and_error('Na-22_gaus_1',  handlers['B_Na'].subt),
    get_sigma_and_error('Na-22_gaus_2',  handlers['B_Na'].subt),
    get_sigma_and_error('Na-22_gaus_3',  handlers['B_Na'].subt),
    get_sigma_and_error('Co-60_gaus_1',  handlers['B_Co'].subt),
    get_sigma_and_error('Co-60_gaus_2',  handlers['B_Co'].subt),
])

PeakPars = namedtuple('PeakPars', ['mean', 'sigma', 'err_mean', 'err_sigma', 'fwhm', 'res'])

fwhm_factor = np.sqrt(8*np.log(2))
src_peaks_pars = {}
peak_names = ['Cs1', 'Na1', 'Na2', 'Na3', 'Co1', 'Co2']
real_peaks_ordered = [*CS137_MEV, *NA22_MEV, *CO60_MEV, *K40_MEV, *TL208_MEV]

for chip, means, err_means, sigs, err_sigs, a, b, err_b in zip(('A', 'B'), (means_A, means_B), (errors_A, errors_B), (sigmas_A, sigmas_B), (errs_sig_A, errs_sig_B), (calibPars['a_A'], calibPars['a_B']), (calibPars['b_A'], calibPars['b_B']), (calibPars['sb_A'], calibPars['sb_B'])):
    # print(chip, means, err_means, sigs, err_sigs, a, b, err_b, '\n')
    for name, m, sm, s, ss, r in zip((peak_names), means, err_means, sigs, err_sigs, real_peaks_ordered):
        mean, err_mean = a + b*m, np.sqrt((b*sm)**2 + (m*err_b)**2)
        sig, err_sig = b*s, np.sqrt((b*ss)**2 + (s*err_b)**2)
        fwhm = fwhm_factor*sig
        res = fwhm/mean
        src_peaks_pars[f'{chip}_{name}'] = PeakPars(mean=mean, sigma=sig, err_mean=err_mean, err_sigma=err_sig, fwhm=fwhm, res=res*100)

for k, v in src_peaks_pars.items():
    print(f'{k}:\n {v}\n')

A_Cs1:
 PeakPars(mean=np.float64(0.6175618771921318), sigma=0.13289816206626578, err_mean=np.float64(0.022004055239072298), err_sigma=np.float64(0.004189352339921876), fwhm=np.float64(0.3129512559814144), res=np.float64(50.67528737432914))

A_Na1:
 PeakPars(mean=np.float64(0.4873988513117771), sigma=0.12946963193091518, err_mean=np.float64(0.017950991515153815), err_sigma=np.float64(0.0044657859022773855), fwhm=np.float64(0.3048776844936981), res=np.float64(62.551990771655575))

A_Na2:
 PeakPars(mean=np.float64(1.3364437015625477), sigma=0.13392071761516958, err_mean=np.float64(0.04457058949991003), err_sigma=np.float64(0.004352019334181896), fwhm=np.float64(0.31535919028513065), res=np.float64(23.5968930016594))

A_Na3:
 PeakPars(mean=np.float64(1.7990490818312446), sigma=0.13680227690686908, err_mean=np.float64(0.05928541995090265), err_sigma=np.float64(0.009562445049908391), fwhm=np.float64(0.32214474386616987), res=np.float64(17.906389943417224))

A_Co1:
 PeakPars(mean=np.float64(1

In [26]:
resolutions = {'A':[], 'B':[]}

for k, v in src_peaks_pars.items():
    resolutions[k[0]].append(v.res)

# Add the background peaks data
real_peaks_bg = [K40_MEV[0], TL208_MEV[0]]
res_bg = {'A': [hist_bgA_calib.GetFunction(fnc).GetParameter('Sigma') for fnc in ['K_peak', 'Tl_peak']],
         'B': [hist_bgB_calib.GetFunction(fnc).GetParameter('Sigma') for fnc in ['K_peak', 'Tl_peak']]}
# print(res_bg)

means_bg = {'A':[hist_bgA_calib.GetFunction(fnc).GetParameter('Mean') for fnc in ['K_peak', 'Tl_peak']], 
            'B':[hist_bgB_calib.GetFunction(fnc).GetParameter('Mean') for fnc in ['K_peak', 'Tl_peak']]}

for k, v in res_bg.items():
    # print(k, v)
    res_bg[k] = [fwhm_factor*100*sig/r for sig, r in zip(v, means_bg[k])]

resolutions['A'].extend(res_bg['A'])
resolutions['B'].extend(res_bg['B'])

print(f'{resolutions=}')

resolutions={'A': [np.float64(50.67528737432914), np.float64(62.551990771655575), np.float64(23.5968930016594), np.float64(17.906389943417224), np.float64(27.548683223480726), np.float64(12.532652565502683), np.float64(19.506662029652716), np.float64(7.944074852142248)], 'B': [np.float64(48.72427314558983), np.float64(63.354086373132915), np.float64(23.287923735146734), np.float64(15.61159100708782), np.float64(28.84402827881869), np.float64(14.693934264846712), np.float64(19.087596691395916), np.float64(6.998977234438429)]}


In [42]:

labels = [
          f"{{}}^{137}Cs {CS137_MEV[0]} ",
          ]

labels

['{}^137Cs 0.661 ']

In [59]:
real_peaks_ordered = np.array(real_peaks_ordered)
resolutions['A'] = np.array(resolutions['A'])
resolutions['B'] = np.array(resolutions['B'])

labels = [
    "{}^{137}Cs " + f"{CS137_MEV[0]:.2f} MeV",
    "{}^{22}Na " + f"{NA22_MEV[0]:.2f} MeV",
    "{}^{22}Na " + f"{NA22_MEV[1]:.2f} MeV",
    "{}^{22}Na " + f"{NA22_MEV[2]:.2f} MeV",
    "{}^{60}Co " + f"{CO60_MEV[0]:.2f} MeV",
    "{}^{60}Co " + f"{CO60_MEV[1]:.2f} MeV",
    "{}^{40}K "  + f"{K40_MEV[0]:.2f} MeV",
    "{}^{208}Tl "+ f"{TL208_MEV[0]:.2f} MeV",
          ]

if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 800, 600)
lg = ROOT.TLegend(0.48, 0.61, 0.75, 0.83)

zeros = np.zeros(len(means_A), dtype=np.float64)

res_graph_A = ROOT.TGraph(8, real_peaks_ordered, resolutions['A'])
res_graph_A.SetMarkerColor(COLOR_A)
res_graph_A.SetMarkerStyle(20)
correction_B = 1
res_graph_B = ROOT.TGraph(8, real_peaks_ordered, resolutions['B'])
res_graph_B.SetMarkerColor(COLOR_B)
res_graph_B.SetMarkerStyle(20)

res_graph_A.SetTitle("Energy resolution")
res_graph_A.GetXaxis().SetTitle("Energy (MeV)")
res_graph_A.GetYaxis().SetTitle("Resolution (%)")

lg.AddEntry(res_graph_A, "Detector A")
lg.AddEntry(res_graph_B, "Detector B")
res_graph_A.Draw("AP")
res_graph_B.Draw("P same")

# Add labels
latex = ROOT.TLatex()
latex.SetTextSize(0.02)
latex.SetTextAlign(12)  # alineación: horizontal izquierda, vertical centrada

for i in range(len(resolutions['A'])):
    latex.DrawLatex(real_peaks_ordered[i] + 0.05, resolutions['A'][i], labels[i])  # pequeño offset en x


lg.Draw()
cv.Update()
cv.Draw()

In [28]:
res_trend_A = ROOT.TF1('res_trend_A', '[0]/TMath::Sqrt(x)+[1]', 0, 3)
res_graph_A.Fit(res_trend_A, 'RS 0')
res_trend_B = ROOT.TF1('res_trend_B', '[0]/TMath::Sqrt(x)+[1]', 0, 3)
res_graph_B.Fit(res_trend_B, 'RS 0')

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      31.4471
NDf                       =            6
Edm                       =  8.30055e-21
NCalls                    =           29
p0                        =      68.3638   +/-   3.12316     
p1                        =     -34.0379   +/-   2.93794     
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =        64.63
NDf                       =            6
Edm                       =  1.50469e-20
NCalls                    =           29
p0                        =       68.036   +/-   4.47735     
p1                        =      -33.949   +/-   4.21182     


In [61]:
ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 800, 600)
lg = ROOT.TLegend(0.65, 0.75, 0.86, 0.88)

lg.AddEntry(res_graph_A, "Detector A")
lg.AddEntry(res_graph_B, "Detector B")

res_trend_A.SetLineColor(COLOR_A)
res_trend_A.SetLineStyle(2)
res_trend_B.SetLineColor(COLOR_B)
res_trend_B.SetLineStyle(2)

res_graph_A.Draw("AP")
res_graph_B.Draw("P same")
res_trend_A.Draw("same")
res_trend_B.Draw("same")
lg.Draw()

# get parameters
fit_pars_res = {}
eqboxes = {}
for counter, chip, fitfn in zip([0,1], ['A', 'B'], [res_trend_A, res_trend_B]):
    eqboxes[f'{chip}'] = ROOT.TPaveText(0.5, 0.6-counter*(0.13), 0.86, 0.7-counter*(0.13), "NDC")
    box =  eqboxes[f'{chip}']
    box.SetFillColor(0)
    box.SetBorderSize(1)
    a = fitfn.GetParameter(0)
    b = fitfn.GetParameter(1)
    sa = fitfn.GetParError(0)
    sb = fitfn.GetParError(1)

    fit_pars_res[f'a_{chip}'] = a
    fit_pars_res[f'b_{chip}'] = b
    fit_pars_res[f'sa_{chip}'] = sa
    fit_pars_res[f'sb_{chip}'] = sb

    box.AddText(f"Detector {chip}")
    box.AddText(f"R = ({a:.4f} #pm {sa:.4f})*{'E^{-1/2}'} + ({b:.4f} #pm {sb:.4f})")
    box.Draw()

# Add labels
latex = ROOT.TLatex()
latex.SetTextSize(0.02)
latex.SetTextAlign(12)  # alineación: horizontal izquierda, vertical centrada

for i in range(len(resolutions['A'])):
    latex.DrawLatex(real_peaks_ordered[i] + 0.05, resolutions['A'][i], labels[i])  # pequeño offset en x


cv.Draw()

# LED

In [30]:
# Files
acq_nrs = [_ for _ in range(11, 23)]
led_files_A = sorted([list((project_root/'data/260519').glob(f'SUBT_A{nr}*.csv'))[0] for nr in acq_nrs])
led_files_B = sorted([list((project_root/'data/260519').glob(f'SUBT_B{nr}*.csv'))[0] for nr in acq_nrs])

hiLs = [3., 2.9, 2.8995, 2.9, 2.85, 2.84, 2.83, 2.82, 2.91, 2.93, 2.95]

led_files_B

[PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B11_Cs137_LED80ns3V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B12_Cs137_LED80ns2,9V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B13_Cs137_LED80ns2,8995V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B14_Cs137_LED80ns2,9V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B15_Cs137_LED80ns2,85V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B16_Cs137_LED80ns2,84V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B17_Cs137_LED80ns2,83V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B18_Cs137_LED80ns2,82V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B19_Cs137_LED80ns2,91V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B20_Cs137_LED80ns2,92V.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/

In [31]:
# Acquisitions, histograms

led_acqs_A = [SiphraAcquisition(file) for file in led_files_A]
led_acqs_B = [SiphraAcquisition(file) for file in led_files_B]

led_acqs_B

[SIPHRAAcquisition(File: 'SUBT_B11_Cs137_LED80ns3V'),
 SIPHRAAcquisition(File: 'SUBT_B12_Cs137_LED80ns2,9V'),
 SIPHRAAcquisition(File: 'SUBT_B13_Cs137_LED80ns2,8995V'),
 SIPHRAAcquisition(File: 'SUBT_B14_Cs137_LED80ns2,9V'),
 SIPHRAAcquisition(File: 'SUBT_B15_Cs137_LED80ns2,85V'),
 SIPHRAAcquisition(File: 'SUBT_B16_Cs137_LED80ns2,84V'),
 SIPHRAAcquisition(File: 'SUBT_B17_Cs137_LED80ns2,83V'),
 SIPHRAAcquisition(File: 'SUBT_B18_Cs137_LED80ns2,82V'),
 SIPHRAAcquisition(File: 'SUBT_B19_Cs137_LED80ns2,91V'),
 SIPHRAAcquisition(File: 'SUBT_B20_Cs137_LED80ns2,92V'),
 SIPHRAAcquisition(File: 'SUBT_B21_Cs137_LED80ns2,93V'),
 SIPHRAAcquisition(File: 'SUBT_B22_Cs137_LED80ns2,95V')]

In [32]:
handlers

{'A_BG': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A01_NewSourceTest_Background'), hist=<cppyy.gbl.TH1F object at 0x5f10dfc0f150>, src='Background', subt=None),
 'A_Cs': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A02_NewSourceTest_Cs137'), hist=<cppyy.gbl.TH1F object at 0x5f10e0143cd0>, src='Cs-137', subt=<cppyy.gbl.TH1F object at 0x5f10e28e5a10>),
 'A_Na': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A03_NewSourceTest_Na22'), hist=<cppyy.gbl.TH1F object at 0x5f10d96d21e0>, src='Na-22', subt=<cppyy.gbl.TH1F object at 0x5f10e211dc70>),
 'A_Co': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A04_NewSourceTest_Co60'), hist=<cppyy.gbl.TH1F object at 0x5f10da330620>, src='Co-60', subt=<cppyy.gbl.TH1F object at 0x5f10e1ff42e0>),
 'B_BG': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_B01_NewSourceTest_Background'), hist=<cppyy.gbl.TH1F object at 0x5f10df428990>, src='Background', subt=None),
 'B_Cs': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_B02_NewSourceTest_Cs137'), hist=<cppyy.g

In [33]:
CS137_MEV

[0.661]